# NASA C-MAPSS FD001 Turbofan Engine Predictive Maintenance (PdM)
## Machine Learning & Deep Learning Implementation for Remaining Useful Life (RUL)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Suksha128/nasa_turbofan/blob/main/notebooks/nasa_cmapss_pdm_model.ipynb)

---

### 🛫 Executive Problem & Motivation
In commercial aviation, high-bypass turbofan engines (e.g., GE90, CFM56, Trent 700) cost **$15M–$30M**. Traditional scheduled maintenance overhauls healthy engines prematurely (wasting **$450,000+** per shop visit) or risks catastrophic In-Flight Shutdowns (IFSD).

This notebook implements an end-to-end Machine Learning and Deep Learning pipeline on the **NASA C-MAPSS FD001 dataset**:
1. **Piecewise Linear RUL Target:** Clamped at 125 cycles following NASA benchmark practice.
2. **Zero-Variance Channel Pruning:** Eliminating 7 constant, uninformative sensor channels.
3. **Temporal Feature Engineering:** 5-cycle rolling statistics to reject aerodynamic turbulence.
4. **NASA Asymmetric Scoring Function:** Evaluating models with asymmetric exponential penalties where late predictions are penalized **1.75×+** more than early warnings.
5. **Multiple Model Architectures:** Random Forest, XGBoost, and Deep Learning LSTM.
6. **Explainability & Visualizations:** Feature importance, actual vs. predicted curves, and loss comparison.

### 1. Environment Setup & Library Imports

In [ ]:
# Install xgboost and torch if not present
!pip install -q xgboost scikit-learn matplotlib seaborn torch

import os
import sys
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries loaded successfully! PyTorch version:", torch.__version__)

### 2. Dataset Ingestion (NASA C-MAPSS FD001)
Download the official dataset files: `train_FD001.txt`, `test_FD001.txt`, and `RUL_FD001.txt`.
Includes an automated mirror fallback to ensure 100% reliable execution in Colab.

In [ ]:
# Create data directory
os.makedirs("data", exist_ok=True)

# Download official C-MAPSS dataset from reliable GitHub archive
base_url = "https://raw.githubusercontent.com/Azure/azure-sdk-for-python/main/sdk/ml/azure-ai-ml/tests/test_configs/dataset/cmapss/"
fallback_url = "https://raw.githubusercontent.com/hankroark/Turbofan-Engine-Degradation/master/"

files = ["train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt"]
for f in files:
    dest = os.path.join("data", f)
    if not os.path.exists(dest):
        try:
            print(f"Downloading {f}...")
            urllib.request.urlretrieve(f"{base_url}{f}", dest)
        except Exception:
            try:
                urllib.request.urlretrieve(f"{fallback_url}{f}", dest)
            except Exception as e:
                print(f"Could not download {f}: {e}")

# Standard C-MAPSS column schema (26 columns: 1 unit + 1 cycle + 3 settings + 21 sensors)
columns = ['unit_nr', 'time_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + [f's_{i}' for i in range(1, 22)]

train_df = pd.read_csv('data/train_FD001.txt', sep=r'\s+', header=None, names=columns)
test_df = pd.read_csv('data/test_FD001.txt', sep=r'\s+', header=None, names=columns)
rul_df = pd.read_csv('data/RUL_FD001.txt', sep=r'\s+', header=None, names=['RUL_ground_truth'])

print(f"Training set shape: {train_df.shape} ({train_df['unit_nr'].nunique()} engines)")
print(f"Test set shape:     {test_df.shape} ({test_df['unit_nr'].nunique()} engines)")
print(f"Ground truth RUL shape: {rul_df.shape}")
train_df.head(3)

### 3. Target Label Generation & Piecewise Linear Clamping
In run-to-failure data, ground truth RUL at cycle $t$ is:
$$\text{RUL}(t) = \text{MaxCycle} - t$$

#### The Piecewise Linear Cap (125 Cycles):
During early engine life, there is negligible mechanical wear. Differentiating RUL 250 vs 200 is statistical noise that harms gradient convergence. Following Heimes (2008), we clip RUL to a maximum of **125 cycles**:
$$\text{RUL}_{\text{clipped}}(t) = \min(125, \, \text{RUL}(t))$$

In [ ]:
# Calculate linear true RUL
max_cycle_per_unit = train_df.groupby('unit_nr')['time_cycles'].transform('max')
train_df['RUL'] = max_cycle_per_unit - train_df['time_cycles']

# Apply Piecewise Linear Clamping (RUL_CAP = 125 cycles)
RUL_CAP = 125
train_df['RUL_clipped'] = train_df['RUL'].clip(upper=RUL_CAP)

# Visualize RUL clipping for Engine #1
plt.figure(figsize=(10, 4))
engine_1 = train_df[train_df['unit_nr'] == 1]
plt.plot(engine_1['time_cycles'], engine_1['RUL'], label='Raw Linear RUL', color='gray', linestyle='--')
plt.plot(engine_1['time_cycles'], engine_1['RUL_clipped'], label=f'Piecewise Capped RUL (max={RUL_CAP})', color='#0284c7', linewidth=2.5)
plt.axvline(x=engine_1['time_cycles'].max() - RUL_CAP, color='#ef4444', linestyle=':', label='Wear Knee-Point Onset')
plt.title('Engine #1: Raw RUL vs Piecewise Linear Clamped Target', fontsize=13, fontweight='bold')
plt.xlabel('Operational Flight Cycles', fontsize=11)
plt.ylabel('RUL Target (Cycles)', fontsize=11)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

### 4. Zero-Variance Sensor Pruning & Feature Engineering

Under steady-state sea-level conditions (FD001), **7 sensors have zero variance across all engines**:  
`s_1, s_5, s_6, s_10, s_16, s_18, s_19`  
We prune these flatline sensors to prevent collinearity.

Then, we compute **5-cycle trailing rolling statistics (Mean $\mu$ and Std $\sigma$)** to smooth aerodynamic turbulence and capture degradation trends.

In [ ]:
# 1. Identify and drop zero-variance / constant sensors
sensor_cols = [f's_{i}' for i in range(1, 22)]
low_var_sensors = [col for col in sensor_cols if train_df[col].std() < 1e-4]
print("Pruned Zero-Variance Channels:", low_var_sensors)

active_sensors = [col for col in sensor_cols if col not in low_var_sensors]
print(f"Retained {len(active_sensors)} active degradation sensors: {active_sensors}")

# 2. Feature Engineering: Trailing 5-cycle rolling statistics
def add_rolling_features(df, sensor_list, window=5):
    df_out = df.copy()
    for s in sensor_list:
        # Rolling mean (smoothed trajectory)
        df_out[f'{s}_mean'] = df_out.groupby('unit_nr')[s].rolling(window, min_periods=1).mean().reset_index(0, drop=True)
        # Rolling standard deviation (volatility/turbulence)
        df_out[f'{s}_std'] = df_out.groupby('unit_nr')[s].rolling(window, min_periods=1).std().fillna(0).reset_index(0, drop=True)
    return df_out

train_feat = add_rolling_features(train_df, active_sensors, window=5)
test_feat = add_rolling_features(test_df, active_sensors, window=5)

feature_cols = active_sensors + [f'{s}_mean' for s in active_sensors] + [f'{s}_std' for s in active_sensors]
print(f"Total engineered feature dimension: {len(feature_cols)}")

# 3. Normalization using MinMaxScaler
scaler = MinMaxScaler()
train_feat[feature_cols] = scaler.fit_transform(train_feat[feature_cols])
test_feat[feature_cols] = scaler.transform(test_feat[feature_cols])

# Prepare training matrices
X_train = train_feat[feature_cols].values
y_train = train_feat['RUL_clipped'].values

# For testing, the benchmark task predicts RUL at the LAST recorded cycle for each test engine
test_last_records = test_feat.groupby('unit_nr').last().reset_index()
X_test = test_last_records[feature_cols].values
# Ground truth for test units is RUL_FD001.txt (clipped at RUL_CAP for consistency)
y_test_raw = rul_df['RUL_ground_truth'].values
y_test = np.clip(y_test_raw, 0, RUL_CAP)

print(f"Training set: X={X_train.shape}, y={y_train.shape}")
print(f"Test evaluation set (100 engines): X={X_test.shape}, y={y_test.shape}")

### 5. Evaluation Metrics: NASA Asymmetric Scoring vs RMSE

The **NASA Asymmetric Scoring Function** penalizes prediction error $d = \hat{y} - y$:
$$s_i = \begin{cases} \exp(-d_i / 13) - 1 & \text{if } d_i < 0 \text{ (Early, Safe)} \\ \exp(d_i / 10) - 1 & \text{if } d_i \ge 0 \text{ (Late, Catastrophic)} \end{cases}$$

Late predictions are penalized exponentially harder to enforce aviation safety.

In [ ]:
def calculate_nasa_score(y_true, y_pred):
    """Calculates NASA Asymmetric Scoring Function penalty."""
    d = y_pred - y_true
    penalty = np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)
    return float(np.sum(penalty))

def evaluate_model(y_true, y_pred, model_name="Model"):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    nasa_score = calculate_nasa_score(y_true, y_pred)
    
    print(f"\n{'='*20} {model_name} {'='*20}")
    print(f"  RMSE:        {rmse:8.3f} cycles")
    print(f"  MAE:         {mae:8.3f} cycles")
    print(f"  R² Score:    {r2:8.3f}")
    print(f"  NASA Score:  {nasa_score:8.2f}")
    return {"model": model_name, "rmse": rmse, "mae": mae, "r2": r2, "nasa_score": nasa_score}

# Demonstrate the asymmetry curve
d_vals = np.linspace(-30, 30, 200)
nasa_penalties = np.where(d_vals < 0, np.exp(-d_vals / 13.0) - 1.0, np.exp(d_vals / 10.0) - 1.0)
mse_penalties = 0.05 * (d_vals ** 2)

plt.figure(figsize=(9, 4.5))
plt.plot(d_vals, nasa_penalties, label='NASA Asymmetric Loss (Late Penalty > Early)', color='#ef4444', linewidth=2.5)
plt.plot(d_vals, mse_penalties, label='Symmetric Quadratic Loss (MSE scaled)', color='#3b82f6', linestyle='--', linewidth=2)
plt.axvline(0, color='gray', linestyle=':', alpha=0.7)
plt.annotate('Late Prediction\n(Flight Hazard)', xy=(18, 5), xytext=(12, 10), arrowprops=dict(facecolor='#ef4444', shrink=0.05))
plt.annotate('Early Prediction\n(Safe Premature Check)', xy=(-18, 3), xytext=(-28, 8), arrowprops=dict(facecolor='#10b981', shrink=0.05))
plt.title('NASA Asymmetric Score vs Symmetric MSE Loss', fontsize=13, fontweight='bold')
plt.xlabel('Prediction Error d = ŷ - y (cycles)', fontsize=11)
plt.ylabel('Penalty Score', fontsize=11)
plt.ylim(0, 20)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

### 6. Model 1: Random Forest Regressor & XGBoost

In [ ]:
# 1. Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=120, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = np.clip(rf_model.predict(X_test), 0, RUL_CAP)
results_rf = evaluate_model(y_test, rf_pred, model_name="Random Forest")

# 2. XGBoost Regressor
xgb_model = xgb.XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = np.clip(xgb_model.predict(X_test), 0, RUL_CAP)
results_xgb = evaluate_model(y_test, xgb_pred, model_name="XGBoost Regressor")

### 7. Model 2: Deep Learning LSTM Sequence Model (PyTorch)
LSTMs exploit temporal trajectories by taking historical windows of flight cycles (e.g., sequence length = 30) to learn aerothermal degradation patterns.

In [ ]:
# Generate sequence arrays for LSTM (sequence length = 30 cycles)
SEQ_LEN = 30

def create_sequences(df, feature_names, seq_len=30):
    seq_list, target_list = [], []
    for _, unit_df in df.groupby('unit_nr'):
        feat_mat = unit_df[feature_names].values
        rul_vec = unit_df['RUL_clipped'].values
        if len(feat_mat) < seq_len:
            # Pad if shorter than seq_len
            pad_len = seq_len - len(feat_mat)
            feat_mat = np.pad(feat_mat, ((pad_len, 0), (0, 0)), mode='edge')
            rul_vec = np.pad(rul_vec, (pad_len, 0), mode='edge')
        for i in range(len(feat_mat) - seq_len + 1):
            seq_list.append(feat_mat[i : i + seq_len])
            target_list.append(rul_vec[i + seq_len - 1])
    return np.array(seq_list, dtype=np.float32), np.array(target_list, dtype=np.float32)

X_seq_train, y_seq_train = create_sequences(train_feat, feature_cols, seq_len=SEQ_LEN)

# For testing: take the last SEQ_LEN cycles for each unit
X_seq_test_list = []
for _, unit_df in test_feat.groupby('unit_nr'):
    feat_mat = unit_df[feature_cols].values
    if len(feat_mat) < SEQ_LEN:
        pad_len = SEQ_LEN - len(feat_mat)
        feat_mat = np.pad(feat_mat, ((pad_len, 0), (0, 0)), mode='edge')
    X_seq_test_list.append(feat_mat[-SEQ_LEN:])
X_seq_test = np.array(X_seq_test_list, dtype=np.float32)

print("LSTM Sequences: Train X=", X_seq_train.shape, "Test X=", X_seq_test.shape)

# PyTorch LSTM Architecture
class TurbofanLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out.squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lstm_net = TurbofanLSTM(input_dim=len(feature_cols), hidden_dim=64, num_layers=2).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(lstm_net.parameters(), lr=0.002, weight_decay=1e-5)

# Training loop
train_loader = DataLoader(TensorDataset(torch.tensor(X_seq_train), torch.tensor(y_seq_train)), batch_size=128, shuffle=True)
lstm_net.train()
EPOCHS = 20
print(f"Training LSTM on {device} for {EPOCHS} epochs...")
for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    for b_x, b_y in train_loader:
        b_x, b_y = b_x.to(device), b_y.to(device)
        optimizer.zero_grad()
        pred = lstm_net(b_x)
        loss = criterion(pred, b_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(b_x)
    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:2d}/{EPOCHS} - Loss (MSE): {total_loss / len(X_seq_train):.3f}")

# Inference
lstm_net.eval()
with torch.no_grad():
    lstm_pred_tensor = lstm_net(torch.tensor(X_seq_test).to(device))
    lstm_pred = np.clip(lstm_pred_tensor.cpu().numpy(), 0, RUL_CAP)

results_lstm = evaluate_model(y_test, lstm_pred, model_name="PyTorch 2-Layer LSTM")

### 8. Comprehensive Visualizations & Model Benchmarking

In [ ]:
# 1. Model Performance Comparison Table
summary_df = pd.DataFrame([results_rf, results_xgb, results_lstm])
print("\n" + "="*30 + " BENCHMARK SUMMARY " + "="*30)
display(summary_df.sort_values('nasa_score'))

# 2. Plot Actual vs Predicted RUL (Sorted for clarity)
plt.figure(figsize=(14, 5))
sort_idx = np.argsort(y_test)
plt.plot(np.arange(100), y_test[sort_idx], label='Ground Truth RUL', color='black', linewidth=2.5)
plt.plot(np.arange(100), rf_pred[sort_idx], label=f'Random Forest (RMSE: {results_rf["rmse"]:.1f})', color='#0284c7', alpha=0.85)
plt.plot(np.arange(100), xgb_pred[sort_idx], label=f'XGBoost (RMSE: {results_xgb["rmse"]:.1f})', color='#10b981', alpha=0.85)
plt.plot(np.arange(100), lstm_pred[sort_idx], label=f'LSTM (RMSE: {results_lstm["rmse"]:.1f})', color='#f59e0b', linestyle='--', alpha=0.85)
plt.title('Test Fleet (100 Engines): Ground Truth vs Model Predictions (Sorted by RUL)', fontsize=13, fontweight='bold')
plt.xlabel('Test Engine Rank (Sorted by actual lifespan remaining)', fontsize=11)
plt.ylabel('RUL (Flight Cycles)', fontsize=11)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

# 3. Top Feature Importance (XGBoost)
plt.figure(figsize=(10, 5))
importances = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(10)
sns.barplot(x=importances.values, y=importances.index, palette='crest')
plt.title('Top 10 Most Predictive Sensor Features (XGBoost Gini Importance)', fontsize=13, fontweight='bold')
plt.xlabel('Relative Feature Importance Score', fontsize=11)
plt.tight_layout()
plt.show()

### 9. Key Findings & Takeaways
1. **Thermodynamic Sensor Dominance:** High-Pressure Compressor (HPC) Static Pressure (`s_11`), Low-Pressure Turbine (LPT) Temp (`s_4`), and HPC Outlet Temp (`s_3`) carry the vast majority of predictive importance, matching turbofan gas-path physics.
2. **Safety-Aligned Evaluation:** Models optimizing solely for MSE often yield severe NASA scores if errors are late ($d > 0$). Production aviation models must incorporate asymmetric loss penalties to enforce conservative maintenance intervals.
3. **Sequence Models:** LSTMs capture cumulative wear dynamics effectively through multi-cycle temporal windows, helping filter instantaneous transducer noise.